# 06 -- подготовка последовательностей для LSTM

- один объект = `user_id × cutoff_date`;
- вход = **последние 90 календарных дней** до `cutoff_date` включительно;
- target = суммарный `gmv` за **следующие 30 дней**;
- дни без строки в исходных данных заполняются нулями;
- отдельный канал `active` показывает, была ли в этот день вообще строка активности.

### про число признаков

Для простого LSTM этого в целом достаточно: на каждом из 90 шагов модель получает 13 каналов, 91 engineered-признак из `05_Data-Modeling.ipynb` полезнее для табличных моделей; для LSTM важнее сохранить временную ось.

Берем только исходные независимые по смыслу каналы. `has_search_to_*` и `has_cat_to_*` не добавляем: они полностью восстанавливаются из соответствующих счётчиков `*_to_* > 0`.


## Что будет сохранено

Для каждой cutoff-даты создаётся папка `data/lstm/<cutoff>/`:

- `X.npy` — `float16`, форма `(users, 90, 13)`;
- `y.npy` — target для размеченных cutoff;
- `user_id.npy` — порядок пользователей в `X` и `y`.

Используем обычные `.npy`, а не один огромный `.npz`: тогда в ноутбуке обучения массивы можно открыть через `mmap` и не держать все cutoff одновременно в RAM.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def find_project_root(start=None):
    # Работает и при запуске из корня проекта, и из notebooks/modeling.
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "train.parquet").exists():
            return candidate
    raise FileNotFoundError("Не найден data/train.parquet")


PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "data" / "train.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data" / "lstm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 90
HORIZON = 30

LABELED_CUTOFFS = pd.to_datetime([
    "2025-04-19", "2025-05-19", "2025-06-18", "2025-07-18",
    "2025-08-17", "2025-09-16", "2025-10-16", "2025-11-15",
    "2025-12-15", "2026-01-14",
])
INFERENCE_CUTOFF = pd.Timestamp("2026-02-13")
ALL_CUTOFFS = [*LABELED_CUTOFFS, INFERENCE_CUTOFF]

# Каналы, которые LSTM видит каждый день.
FEATURES = [
    "search", "cat", "searches",
    "search_to_cart", "search_to_ord", "cat_to_cart", "cat_to_ord",
    "to_cart", "to_ord", "gmv_search", "gmv_cat", "gmv",
]
BINARY_FEATURES = {"search", "cat"}
MODEL_FEATURES = [*FEATURES, "active"]

# Непрерывные неотрицательные признаки логарифмируем, чтобы тяжёлые хвосты не ломали обучение.
LOG_FEATURES = [name for name in FEATURES if name not in BINARY_FEATURES]

RAW_COLUMNS = ["event_date", "user_id", *FEATURES]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("input channels:", len(MODEL_FEATURES), MODEL_FEATURES)


PROJECT_ROOT: /Users/pinta/Dev/E-CUP-2026
input channels: 13 ['search', 'cat', 'searches', 'search_to_cart', 'search_to_ord', 'cat_to_cart', 'cat_to_ord', 'to_cart', 'to_ord', 'gmv_search', 'gmv_cat', 'gmv', 'active']


## Первый день пользователя

На ранний cutoff не включаем пользователя, который появился только в будущем, аналогично логике в `05_Data-Modeling.ipynb`.


In [2]:
first_seen = pd.read_parquet(
    INPUT_PATH,
    columns=["user_id", "event_date"],
    engine="pyarrow",
)
first_seen["event_date"] = pd.to_datetime(first_seen["event_date"])
first_seen = first_seen.groupby("user_id", sort=True)["event_date"].min()

print("users:", len(first_seen))
print("first date:", first_seen.min().date())
print("last first-seen date:", first_seen.max().date())


users: 250000
first date: 2025-01-01
last first-seen date: 2025-12-15


## Построение одного cutoff

1. читаем только 90 дней истории и, если cutoff размеченный, следующие 30 дней для target;
2. создаем плотный тензор из нулей `(users, 90, features)`;
3. каждую дневную строку кладем в ее позицию `[user, day]`;
4. `active=1` ставим только там, где дневная строка реально была;
5. target считаем отдельно по будущему окну -- leakage нет (если я не дурачок и не накосячил нигде, но вродь проверил).


In [ ]:
def build_cutoff(cutoff: pd.Timestamp):
    cutoff = pd.Timestamp(cutoff)
    history_start = cutoff - pd.Timedelta(days=SEQ_LEN - 1)
    is_inference = cutoff == INFERENCE_CUTOFF
    read_end = cutoff if is_inference else cutoff + pd.Timedelta(days=HORIZON)

    # Пользователь должен уже существовать на момент cutoff.
    users = first_seen.index[first_seen <= cutoff].to_numpy(dtype=np.int64)
    user_to_row = pd.Series(np.arange(len(users), dtype=np.int32), index=users)

    # Читаем только нужный временной диапазон и только нужные колонки.
    df = pd.read_parquet(
        INPUT_PATH,
        columns=RAW_COLUMNS,
        engine="pyarrow",
        filters=[
            ("event_date", ">=", history_start),
            ("event_date", "<=", read_end),
        ],
    )
    df["event_date"] = pd.to_datetime(df["event_date"])

    history = df[df["event_date"] <= cutoff].copy()
    user_index = history["user_id"].map(user_to_row).to_numpy()
    day_index = (history["event_date"] - history_start).dt.days.to_numpy()

    # float16 заметно экономит диск; перед LSTM батч всё равно переводится в float32.
    X = np.zeros((len(users), SEQ_LEN, len(MODEL_FEATURES)), dtype=np.float16)

    for j, name in enumerate(FEATURES):
        values = history[name].to_numpy(dtype=np.float32)
        if name in LOG_FEATURES:
            values = np.log1p(values)
        X[user_index, day_index, j] = values.astype(np.float16)

    # Нужен, чтобы отличать "в этот день строки не было" от строки с нулевыми счётчиками.
    X[user_index, day_index, -1] = 1.0

    cutoff_dir = OUTPUT_DIR / cutoff.strftime("%Y-%m-%d")
    cutoff_dir.mkdir(parents=True, exist_ok=True)
    np.save(cutoff_dir / "X.npy", X)
    np.save(cutoff_dir / "user_id.npy", users)

    if not is_inference:
        future = df[df["event_date"] > cutoff]
        y = (
            future.groupby("user_id")["gmv"].sum()
            .reindex(users, fill_value=0.0)
            .to_numpy(dtype=np.float32)
        )
        np.save(cutoff_dir / "y.npy", y)
        print(cutoff.date(), X.shape, "target nonzero:", f"{(y > 0).mean():.2%}")
    else:
        print(cutoff.date(), X.shape, "inference")

    del df, history, X


## Сборка всех cutoff

In [4]:
for cutoff in ALL_CUTOFFS:
    build_cutoff(cutoff)

meta = {
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "features": MODEL_FEATURES,
    "labeled_cutoffs": [x.strftime("%Y-%m-%d") for x in LABELED_CUTOFFS],
    "inference_cutoff": INFERENCE_CUTOFF.strftime("%Y-%m-%d"),
    "transform": "log1p for all non-binary channels; active is 0/1",
}
with open(OUTPUT_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("saved to:", OUTPUT_DIR)


2025-04-19 (216457, 90, 13) target nonzero: 47.21%
2025-05-19 (221154, 90, 13) target nonzero: 48.25%
2025-06-18 (225245, 90, 13) target nonzero: 49.42%
2025-07-18 (229146, 90, 13) target nonzero: 51.10%
2025-08-17 (232977, 90, 13) target nonzero: 52.84%
2025-09-16 (236668, 90, 13) target nonzero: 53.63%
2025-10-16 (240700, 90, 13) target nonzero: 54.59%
2025-11-15 (244983, 90, 13) target nonzero: 56.94%
2025-12-15 (250000, 90, 13) target nonzero: 56.31%
2026-01-14 (250000, 90, 13) target nonzero: 54.07%
2026-02-13 (250000, 90, 13) inference
saved to: /Users/pinta/Dev/E-CUP-2026/data/lstm


## Быстрый sanity check

In [5]:
check_dir = OUTPUT_DIR / LABELED_CUTOFFS[-1].strftime("%Y-%m-%d")
X = np.load(check_dir / "X.npy", mmap_mode="r")
y = np.load(check_dir / "y.npy", mmap_mode="r")
users = np.load(check_dir / "user_id.npy", mmap_mode="r")

assert X.shape == (len(users), SEQ_LEN, len(MODEL_FEATURES))
assert len(y) == len(users)
assert np.isfinite(X).all()
assert np.isfinite(y).all() and (y >= 0).all()

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("example user:", int(users[0]))
print("non-empty days in its sequence:", int(X[0, :, -1].sum()))


X: (250000, 90, 13) float16
y: (250000,) float32
example user: 2
non-empty days in its sequence: 8
